In [1]:
!pip install ultralytics -q
!pip install pyyaml pandas seaborn -q


import os
import yaml
import shutil
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from google.colab import drive
from sklearn.model_selection import train_test_split

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 16.1 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
BASE = "/content/drive/MyDrive/Thesis_Z"

# Dataset paths
DATASETS = {
    "cloudy": f"{BASE}/cloudy",
    "rainy": f"{BASE}/rainy",
    "snowy": f"{BASE}/snowy"
}

# Working directories
WORK_DIR = "/content/weather_pipeline"
MERGED = f"{WORK_DIR}/merged"
FINAL_DATASET = f"{WORK_DIR}/final_dataset"

os.makedirs(WORK_DIR, exist_ok=True)

Load class list from yml

In [ ]:
dataset_classes = {}

for name, path in DATASETS.items():

    yaml_path = os.path.join(path, "data.yaml")

    with open(yaml_path) as f:
        data = yaml.safe_load(f)

    dataset_classes[name] = data["names"]

    print(name, "classes:")
    print(data["names"])
    print()

build unified class

In [ ]:
INVALID_CLASSES = {"busst", "r", "sp", "objects"}

all_classes = set()

for dataset in dataset_classes.values():

    for c in dataset:

        if c not in INVALID_CLASSES:
            all_classes.add(c)

UNIFIED_CLASSES = sorted(list(all_classes))

print("Unified Classes:\n")

for i,c in enumerate(UNIFIED_CLASSES):
    print(i,"->",c)

print("\nTotal classes:",len(UNIFIED_CLASSES))

Unified Classes:

0 -> bike
1 -> bus
2 -> car
3 -> cycle
4 -> divider
5 -> dog
6 -> fence
7 -> firehydrant
8 -> person
9 -> postbox
10 -> raindroplet
11 -> shrub
12 -> sideWalk
13 -> sign
14 -> snow
15 -> streetlamp
16 -> trashcan
17 -> tree
18 -> truck
19 -> wetground

Total classes: 20


create mapping table

In [ ]:
class_to_id = {c:i for i,c in enumerate(UNIFIED_CLASSES)}

dataset_mapping = {}

for name,classes in dataset_classes.items():

    mapping = {}

    for old_id,c in enumerate(classes):

        if c in INVALID_CLASSES:
            mapping[old_id] = None
        else:
            mapping[old_id] = class_to_id[c]

    dataset_mapping[name] = mapping

print(dataset_mapping)

{'cloudy': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 11, 11: 12, 12: 13, 13: 15, 14: 16, 15: 17, 16: 18}, 'rainy': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 6, 6: 7, 7: None, 8: 8, 9: 10, 10: 13, 11: 15, 12: 16, 13: 17, 14: 18, 15: 19}, 'snowy': {0: 0, 1: 1, 2: None, 3: 2, 4: 3, 5: 4, 6: 5, 7: None, 8: 8, 9: None, 10: 13, 11: 14, 12: None, 13: 15, 14: 16, 15: 17, 16: 18}}


verify dataset structure

In [ ]:
def verify_dataset(path):

    img_count = 0
    lbl_count = 0

    for root,dirs,files in os.walk(path):

        for f in files:

            if f.endswith(".jpg") or f.endswith(".png"):
                img_count+=1

            if f.endswith(".txt"):
                lbl_count+=1

    print(path)
    print("Images:",img_count)
    print("Labels:",lbl_count)
    print()

for d in DATASETS.values():
    verify_dataset(d)

/content/drive/MyDrive/Thesis_Z/cloudy
Images: 5332
Labels: 5344

/content/drive/MyDrive/Thesis_Z/rainy
Images: 3609
Labels: 3611

/content/drive/MyDrive/Thesis_Z/snowy
Images: 4536
Labels: 4538



In [ ]:
for name,path in DATASETS.items():
    count=0
    for root,_,files in os.walk(path):
        for f in files:
            if f.lower().endswith((".jpg",".jpeg",".png")):
                count+=1
    print(name,"images:",count)

cloudy images: 5332
rainy images: 3609
snowy images: 4536


merge dataset + remap labels

In [ ]:
if os.path.exists(MERGED):
    shutil.rmtree(MERGED)

os.makedirs(MERGED+"/images")
os.makedirs(MERGED+"/labels")

img_id = 0

for name,path in DATASETS.items():

    mapping = dataset_mapping[name]

    for root,dirs,files in os.walk(path):
        for f in files:
            if not f.lower().endswith((".jpg",".jpeg",".png")):
                continue

            img_path = os.path.join(root,f)
            # FIX: replace images/ with labels/ to get correct label path
            lbl_path = img_path.replace("/images/", "/labels/").rsplit('.',1)[0] + ".txt"

            if not os.path.exists(lbl_path):
                continue

            new_img = f"{img_id}.jpg"
            new_lbl = f"{img_id}.txt"

            shutil.copy(img_path, MERGED+"/images/"+new_img)

            new_lines=[]

            with open(lbl_path) as lf:
                for line in lf:
                    parts=line.strip().split()
                    if not parts:
                        continue
                    cls=int(parts[0])
                    new_cls = mapping.get(cls)
                    if new_cls is None:
                        continue
                    parts[0] = str(new_cls)
                    new_lines.append(" ".join(parts)+"\n")

            # Skip images without valid labels
            if not new_lines:
                continue

            with open(MERGED+"/labels/"+new_lbl,"w") as nf:
                nf.writelines(new_lines)

            img_id+=1

print("Merged dataset size:",img_id)

Merged dataset size: 13402


verify label integrity

In [ ]:
max_class = len(UNIFIED_CLASSES)-1

errors=0

for f in os.listdir(MERGED+"/labels"):

    path=MERGED+"/labels/"+f

    with open(path) as file:

        for line in file:

            cls=int(line.split()[0])

            if cls>max_class:
                errors+=1

print("Label errors:",errors)

Label errors: 0


split

In [ ]:
images = os.listdir(MERGED+"/images")

train,val = train_test_split(images,test_size=0.3,random_state=42)
val,test = train_test_split(val,test_size=0.33,random_state=42)

print("Train:",len(train))
print("Val:",len(val))
print("Test:",len(test))

Train: 9381
Val: 2694
Test: 1327


final dataset structure

In [ ]:
for split in ["train","val","test"]:

    os.makedirs(f"{FINAL_DATASET}/{split}/images",exist_ok=True)
    os.makedirs(f"{FINAL_DATASET}/{split}/labels",exist_ok=True)

copy files into split

In [ ]:
def move_files(file_list,split):

    for f in file_list:

        img=MERGED+"/images/"+f
        lbl=MERGED+"/labels/"+f.replace(".jpg",".txt")

        shutil.copy(img,f"{FINAL_DATASET}/{split}/images/"+f)
        shutil.copy(lbl,f"{FINAL_DATASET}/{split}/labels/"+f.replace(".jpg",".txt"))

move_files(train,"train")
move_files(val,"val")
move_files(test,"test")

final yml

In [ ]:
yaml_data = {
"path": FINAL_DATASET,
"train": "train/images",
"val": "val/images",
"test": "test/images",
"nc": len(UNIFIED_CLASSES),
"names": UNIFIED_CLASSES
}

yaml_path = FINAL_DATASET+"/data.yaml"

with open(yaml_path,"w") as f:
    yaml.dump(yaml_data,f)

print("YAML created")

YAML created


save final dataset (RUN ONCE)

In [ ]:


DRIVE_FINAL = "/content/drive/MyDrive/Thesis_Z/final_dataset"

if not os.path.exists(DRIVE_FINAL):

    print("Saving final dataset to Google Drive...")

    shutil.copytree(FINAL_DATASET, DRIVE_FINAL)

    print("Dataset saved to:", DRIVE_FINAL)

else:
    print("Dataset already exists in Drive. Skipping save.")

Saving final dataset to Google Drive...
Dataset saved to: /content/drive/MyDrive/Thesis_Z/final_dataset


upload final dataset

In [4]:
FINAL_DATASET = "/content/drive/MyDrive/Thesis_Z/final_dataset"
yaml_path = FINAL_DATASET + "/data.yaml"

print("Using cached dataset:", FINAL_DATASET)

Using cached dataset: /content/drive/MyDrive/Thesis_Z/final_dataset


In [5]:

# Load the data.yaml file
with open(yaml_path, "r") as f:
    data_config = yaml.safe_load(f)

# Update the 'path' to point to the correct Google Drive location
data_config["path"] = FINAL_DATASET

# Save the updated data.yaml file back
with open(yaml_path, "w") as f:
    yaml.dump(data_config, f)

print(f"Updated data.yaml path to: {FINAL_DATASET}")

Updated data.yaml path to: /content/drive/MyDrive/Thesis_Z/final_dataset


In [ ]:
import shutil

DRIVE_DATASET = "/content/drive/MyDrive/Thesis_Z/final_dataset"
LOCAL_DATASET = "/content/final_dataset"

if not os.path.exists(LOCAL_DATASET):
    print("Copying dataset to local disk...")
    shutil.copytree(DRIVE_DATASET, LOCAL_DATASET)

yaml_path = LOCAL_DATASET + "/data.yaml"

print("Dataset ready at:", LOCAL_DATASET)

Copying dataset to local disk...
Dataset ready at: /content/final_dataset


train

In [ ]:
model = YOLO("yolov8m.pt")

model.train(
    data=yaml_path,
    epochs=30,
    imgsz=512,
    batch=8 ,
    optimizer="AdamW",
    cache="disk",
    workers=4,
    seed=42,
    pretrained=True,
    project=WORK_DIR,
    name="weather_detection",
    plots=True,
    save_period=5,      # save checkpoint every 5 epochs
    # resume=True,        # resumes if interrupted
    device=0
    #device="cpu"
)

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/final_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=weather_detection, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=100, 

save train [run it instead of training  each time]

In [ ]:
import shutil

DRIVE_MODEL = "/content/drive/MyDrive/Thesis_Z/best_model.pt"

shutil.copy(
    f"{WORK_DIR}/weather_detection/weights/best.pt",
    DRIVE_MODEL
)

print("Model saved to Drive:", DRIVE_MODEL)

load best model

In [6]:
from ultralytics import YOLO

#best = YOLO(f"{WORK_DIR}/weather_detection/weights/best.pt")
best = YOLO("/content/drive/MyDrive/Thesis_Z/best_model.pt")

print("Best model loaded")

Best model loaded


validation metrix

In [7]:
val_metrics = best.val(data=yaml_path, split="val")

print("Validation Results")
print("mAP50:", val_metrics.box.map50)
print("mAP50-95:", val_metrics.box.map)
print("Precision:", val_metrics.box.mp)
print("Recall:", val_metrics.box.mr)

# F1 score
precision = val_metrics.box.mp
recall = val_metrics.box.mr
f1 = 2*(precision*recall)/(precision+recall)

print("F1 Score:", f1)

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,851,340 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 18.4±40.0 ms, read: 0.1±0.0 MB/s, size: 30.9 KB)
val: Scanning /content/drive/MyDrive/Thesis_Z/final_dataset/val/labels.cache... 2694 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2694/2694 470.8Mit/s 0.0s
val: /content/drive/MyDrive/Thesis_Z/final_dataset/val/images/5494.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/Thesis_Z/final_dataset/val/images/5501.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/Thesis_Z/final_dataset/val/images/5506.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/Thesis_Z/final_dataset/val/images/5535.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/Thesis_Z/final_dataset/val/images/5554.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/Thesis_Z/final_dataset/val/images/5561.jpg: 1 duplicate labels remov

test metrix

In [8]:
test_metrics = best.val(data=yaml_path, split="test")

print("Test Results")
print("mAP50:", test_metrics.box.map50)
print("mAP50-95:", test_metrics.box.map)
print("Precision:", test_metrics.box.mp)
print("Recall:", test_metrics.box.mr)

precision = test_metrics.box.mp
recall = test_metrics.box.mr
f1 = 2*(precision*recall)/(precision+recall)

print("F1 Score:", f1)

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 1.1±1.5 ms, read: 0.1±0.0 MB/s, size: 35.6 KB)
val: Scanning /content/drive/MyDrive/Thesis_Z/final_dataset/test/labels... 1327 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1327/1327 3.3it/s 6:39
val: New cache created: /content/drive/MyDrive/Thesis_Z/final_dataset/test/labels.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 973, len(boxes) = 15346. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 83/83 3.6it/s 23.1s
                   all       1327      15346      0.735      0.634      0.679      0.429
                  bike         56         79      0.794      0.646      0.685       0.35
               

training log

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

results = pd.read_csv(f"{WORK_DIR}/weather_detection/results.csv")

print(results.head())

mAP graph

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(results['epoch'],results['metrics/mAP50(B)'],label="mAP50")
plt.plot(results['epoch'],results['metrics/mAP50-95(B)'],label="mAP50-95")

plt.xlabel("Epoch")
plt.ylabel("mAP")
plt.title("mAP vs Epoch")
plt.legend()
plt.grid()

plt.savefig("map_curve.png", dpi=300)
plt.show()

precision recall graph

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(results['epoch'],results['metrics/precision(B)'],label="Precision")
plt.plot(results['epoch'],results['metrics/recall(B)'],label="Recall")

plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Precision vs Recall")
plt.legend()
plt.grid()

plt.savefig("precision_recall_curve.png", dpi=300)
plt.show()

loss curve

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(results['epoch'],results['train/box_loss'],label="Train Box Loss")
plt.plot(results['epoch'],results['val/box_loss'],label="Val Box Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Box Loss Curve")
plt.legend()
plt.grid()

plt.savefig("loss_curve.png", dpi=300)
plt.show()

confusion metrix

In [ ]:
from PIL import Image
from IPython.display import display

cm_path = f"{WORK_DIR}/weather_detection/confusion_matrix.png"

display(Image.open(cm_path))

import shutil
shutil.copy(cm_path,"confusion_matrix.png")

precision recall curve

In [ ]:
pr_path = f"{WORK_DIR}/weather_detection/PR_curve.png"

display(Image.open(pr_path))

sample prediction

In [9]:
import random
from IPython.display import display

sample_imgs = random.sample(os.listdir(FINAL_DATASET+"/test/images"),15)

for img in sample_imgs:

    path = FINAL_DATASET+"/test/images/"+img

    results = best.predict(path,conf=0.25)

    display(results[0].plot())

Output hidden; open in https://colab.research.google.com to view.